In [1]:
import os
import json
import pickle
import faiss
import numpy as np
from dotenv import load_dotenv

load_dotenv()
# Project Root & Processed Data Paths
PROJECT_ROOT = os.getcwd()
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

PATHS = {
    "chunks": os.path.join(DATA_PROCESSED_DIR, "chunks.json"),
    "faiss_index": os.path.join(DATA_PROCESSED_DIR, "faiss_index.bin"),
    "bm25_index": os.path.join(DATA_PROCESSED_DIR, "bm25_index.pkl")
}

# Load Master Metadata Registry & Indices
with open(PATHS["chunks"], "r", encoding="utf-8") as f:
    chunks = json.load(f)

faiss_index = faiss.read_index(PATHS["faiss_index"])

with open(PATHS["bm25_index"], "rb") as f:
    bm25_index = pickle.load(f)

print(f"Total Chunks in Registry : {len(chunks)}")
print(f"FAISS Index Vectors      : {faiss_index.ntotal}")
print(f"BM25 Corpus Size         : {bm25_index.corpus_size}")

Total Chunks in Registry : 1968
FAISS Index Vectors      : 1968
BM25 Corpus Size         : 1968


In [2]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print(f"Loading Bi-Encoder ({BI_ENCODER_NAME}) on {DEVICE}...")
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)

print(f"Loading Cross-Encoder ({CROSS_ENCODER_NAME}) on {DEVICE}...")
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, max_length=512, device=DEVICE)


Loading Bi-Encoder (sentence-transformers/all-MiniLM-L6-v2) on cuda...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading Cross-Encoder (cross-encoder/ms-marco-MiniLM-L-6-v2) on cuda...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [3]:
def hybrid_retrieve_and_rerank(query, candidate_k=20, final_top_k=5, rrf_k=60):
    """
    Stage 1: Parallel Dense (FAISS) + Sparse (BM25) Top-20 Retrieval
    Stage 2: Reciprocal Rank Fusion (RRF) Candidate Merging
    Stage 3: Cross-Encoder Joint Attention Re-Ranking to Top-5
    """
    # 1. Dense Retrieval (FAISS)
    query_embedding = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    dense_scores, dense_indices = faiss_index.search(query_embedding, candidate_k)
    dense_ranked_ids = dense_indices[0].tolist()

    # 2. Sparse Retrieval (BM25)
    tokenized_query = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_query)
    sparse_ranked_ids = np.argsort(bm25_scores)[::-1][:candidate_k].tolist()

    # 3. Reciprocal Rank Fusion (RRF)
    rrf_scores = {}
    for rank, doc_idx in enumerate(dense_ranked_ids):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + (1.0 / (rrf_k + rank + 1))

    for rank, doc_idx in enumerate(sparse_ranked_ids):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + (1.0 / (rrf_k + rank + 1))

    # Select Top-20 fused candidates
    fused_ids = sorted(rrf_scores.keys(), key=lambda idx: rrf_scores[idx], reverse=True)[:candidate_k]

    # 4. Cross-Encoder Re-Ranking
    candidate_pairs = [(query, chunks[idx]["chunk_text"]) for idx in fused_ids]
    cross_scores = cross_encoder.predict(candidate_pairs)

    # Combine & sort by Cross-Encoder score descending
    reranked_results = []
    for idx, ce_score in zip(fused_ids, cross_scores):
        chunk_data = chunks[idx].copy()
        chunk_data["rrf_score"] = round(float(rrf_scores[idx]), 4)
        chunk_data["cross_encoder_score"] = round(float(ce_score), 4)
        reranked_results.append(chunk_data)

    reranked_results.sort(key=lambda x: x["cross_encoder_score"], reverse=True)
    return reranked_results[:final_top_k]



In [4]:
import time

test_queries = [
    "Who is the Director of IIIT Nagpur and what is his contact number?",
    "CSL-422 Machine Learning syllabus or faculty",
    "Hostel rules and fee structure details"
]

for q in test_queries:
    start_time = time.time()
    results = hybrid_retrieve_and_rerank(q, candidate_k=20, final_top_k=5)
    latency_ms = (time.time() - start_time) * 1000

    print("=" * 80)
    print(f"QUERY: {q}  |  Latency: {latency_ms:.1f} ms")
    print("=" * 80)

    for rank, res in enumerate(results, start=1):
        # Strip the prepended metadata header for clean preview display
        body_preview = res["chunk_text"].split("\n\n", 1)[-1][:180].replace("\n", " ")
        print(f"Rank {rank} | CE Score: {res['cross_encoder_score']} | RRF: {res['rrf_score']}")
        print(f"   Title  : {res['title']} ({res['category']})")
        print(f"   Source : {res['source_url']}")
        print(f"   Preview: {body_preview}...\n")

QUERY: Who is the Director of IIIT Nagpur and what is his contact number?  |  Latency: 5364.4 ms
Rank 1 | CE Score: 5.3348 | RRF: 0.0161
   Title  : Indian Institute of Information Technology Nagpur (Governance)
   Source : https://www.iiitn.ac.in/governance/administration
   Preview: about Chairman director Dr. Prem Lal Patel director 📧 Email: director@iiitn.ac.in Professor Prem Lal Patel, Director, VNIT, Nagpur took over the additional charge as Director of II...

Rank 2 | CE Score: 3.4317 | RRF: 0.0147
   Title  : Indian Institute of Information Technology Nagpur (Governance)
   Source : https://www.iiitn.ac.in/governance/committee
   Preview: Committee Sl. No. Name Position 1 Professor (Dr.) Prem Lal Patel Director, IIIT Nagpur Chairman 2 Prof. Tom Mathew Professor (HAG), Dept. of Civil Engineering, IIT Bombay MoE Nomin...

Rank 3 | CE Score: 3.4296 | RRF: 0.0156
   Title  : Read More → (General)
   Source : https://iiitn.ac.in/media/1ba7243672ec400aa6a1ffbc648cadf1.pdf
   Preview:

In [7]:
import os
from typing import List
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_ollama import ChatOllama

class CampusNLPHybridRetriever(BaseRetriever):
    candidate_k: int = 20
    final_top_k: int = 5
    ce_threshold: float = -1.0

    def _get_relevant_documents(self, query: str) -> List[Document]:
        raw_results = hybrid_retrieve_and_rerank(
            query=query,
            candidate_k=self.candidate_k,
            final_top_k=self.final_top_k
        )

        # Filter negative-logit boilerplate noise while always keeping Rank 1
        filtered = [
            r for i, r in enumerate(raw_results)
            if i == 0 or r["cross_encoder_score"] >= self.ce_threshold
        ]

        docs = []
        for idx, r in enumerate(filtered, start=1):
            docs.append(
                Document(
                    page_content=r["chunk_text"],
                    metadata={
                        "source_num": idx,
                        "chunk_id": r["chunk_id"],
                        "title": r["title"],
                        "category": r["category"],
                        "source_url": r["source_url"],
                        "rrf_score": r["rrf_score"],
                        "cross_encoder_score": r["cross_encoder_score"]
                    }
                )
            )
        return docs


# 3. Initialize Retriever & Local Qwen 2.5 via ChatOllama
retriever = CampusNLPHybridRetriever(candidate_k=20, final_top_k=5, ce_threshold=-1.0)

llm = ChatOllama(
    model="qwen2.5:latest",
    temperature=0.1,
    num_ctx=4096
)

# 4. Grounded Prompt & LCEL Chain with Source Attribution

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are CampusNLP, an intelligent and reliable knowledge
assistant for the Indian Institute of Information Technology (IIIT) Nagpur.

Your responsibility is to answer questions about the institute using
the retrieved document passages provided in the context.

STRICT GROUNDING RULES:

1. Use ONLY the information explicitly available in the provided context.
   Never use external knowledge, assumptions, or fabricated information.

2. If the retrieved passages do not contain sufficient information
   to answer the question, respond with:
   "The available IIIT Nagpur documents do not contain enough
   information to answer this question."

3. If multiple passages contain relevant information, combine them
   into one coherent answer. Do not ignore relevant evidence.

4. If sources provide conflicting information:
   - Clearly identify the inconsistency.
   - Do not arbitrarily choose one answer.
   - Mention the relevant source citations.

5. Do not assume that older information is still valid.
   If dates or academic years are mentioned, preserve them exactly.
   Never infer current information from outdated documents.

6. Distinguish between explicitly stated facts and information
   that cannot be established from the context.

ANSWERING GUIDELINES:

1. Understand the complete intent of the user's question before answering.

2. Answer directly and clearly. Avoid unnecessary introductions,
   repetition, and generic explanations.

3. For questions involving multiple parts, address every part
   separately.

4. For structured information such as fee structures, academic
   schedules, course credits, and eligibility criteria:
   - Prefer bullet points or Markdown tables.
   - Preserve exact values, units, dates, and terminology.
   - Do not omit important conditions or exceptions.

5. For simple factual questions, provide concise answers.
   For complex questions, provide sufficient explanation.

SOURCE ATTRIBUTION:

1. Every factual claim must be supported by the retrieved context.

2. Cite sources inline using their assigned source numbers,
   for example [Source 1] or [Source 2].

3. When multiple sources support a statement, cite all relevant
   sources.

4. Do not create source numbers or cite sources that do not exist
   in the provided context.

5. At the end, include a "Sources" section containing the relevant
   source titles and URLs exactly as provided in the context.

6. Never invent, modify, or fabricate URLs.

RESPONSE STYLE:

- Maintain a professional, informative, student-friendly tone.
- Use clear and understandable English.
- Avoid unsupported claims.
- Prioritize factual accuracy over producing an answer.

Remember: Your primary objective is to provide accurate,
document-grounded answers rather than answering every question
at any cost.
"""),

    ("human", """RETRIEVED CONTEXT:

{context}

USER QUESTION:
{question}

Provide a clear, accurate, and fully grounded answer
with appropriate source citations.""")
])



def format_docs_with_sources(docs: List[Document]) -> str:
    blocks = []
    for doc in docs:
        src_num = doc.metadata["source_num"]
        blocks.append(f"[Source {src_num}]\n{doc.page_content}")
    return "\n\n---\n\n".join(blocks)


rag_chain = (
    RunnableParallel(
        docs=retriever,
        question=RunnablePassthrough()
    )
    .assign(
        answer=(
                RunnablePassthrough.assign(context=lambda x: format_docs_with_sources(x["docs"]))
                | prompt
                | llm
                | StrOutputParser()
        )
    )
)

# 5. Run Test Queries & Log Traces to LangSmith
sample_queries = [
    "Who is the Director of IIIT Nagpur and what is his email?",
    "What are the credits and L-T-P structure for CSL-422 Machine Learning?",
    "What is the hostel fee structure for 2026-27?"
]

for q in sample_queries:
    result = rag_chain.invoke(q)
    print("=" * 80)
    print(f"QUESTION: {result['question']}\n")
    print(result["answer"])
    print("\nVerified Sources:")
    seen_urls = set()
    for d in result["docs"]:
        url = d.metadata["source_url"]
        if url not in seen_urls:
            print(
                f" - [Source {d.metadata['source_num']}] {d.metadata['title']} (CE: {d.metadata['cross_encoder_score']}): {url}")
            seen_urls.add(url)
    print("=" * 80 + "\n")

QUESTION: Who is the Director of IIIT Nagpur and what is his email?

The Director of IIIT Nagpur is Professor Prem Lal Patel. His email is director@iiitn.ac.in [Source 1]

Verified Sources:
 - [Source 1] Indian Institute of Information Technology Nagpur (CE: 7.1661): https://www.iiitn.ac.in/governance/administration
 - [Source 2] Read More → (CE: 4.9713): https://iiitn.ac.in/media/1ba7243672ec400aa6a1ffbc648cadf1.pdf
 - [Source 3] Read More → (CE: 3.8589): https://iiitn.ac.in/media/4408aa0356994798827a56612b398256.pdf
 - [Source 4] Indian Institute of Information Technology Nagpur (CE: 2.7826): https://www.iiitn.ac.in/pages/studentachievements
 - [Source 5] Indian Institute of Information Technology Nagpur (CE: 1.3188): https://www.iiitn.ac.in/governance/313

QUESTION: What are the credits and L-T-P structure for CSL-422 Machine Learning?

The credits and L-T-P (Lecture-Tutorial-Practical) structure for CSL-422 Machine Learning is as follows:
- Credits: 4
- L-T-P: 3-0-2

[Source 2]

Ve

## Evaluation

In [9]:
import os
import json
from pathlib import Path

EVAL_DATASET_PATH = os.path.join(PROJECT_ROOT, "data", "eval_dataset.json")

# Ground-truth evaluation benchmark queries for IIIT Nagpur RAG system
benchmark_data = [
    {
        "query_id": "q1",
        "category": "Institute",
        "query": "Who is the Director of IIIT Nagpur and what is his email?",
        "ground_truth_doc_ids": ["html_b39f2948e2"],
        "expected_answer": "Prof. (Dr.) Prem Lal Patel is the Director of IIIT Nagpur. Email: director@iiitn.ac.in",
        "is_answerable": True
    },
    {
        "query_id": "q2",
        "category": "Academics",
        "query": "Who is the HOD of Computer Science and Engineering?",
        "ground_truth_doc_ids": ["pdf_fb94fbf05b774a7e80dfef4a125c4dcb"],
        "expected_answer": "Dr. Milind Penurkar is the HOD of Computer Science & Engineering.",
        "is_answerable": True
    },
    {
        "query_id": "q3",
        "category": "Admissions",
        "query": "What is the contact email for the Registrar of IIIT Nagpur?",
        "ground_truth_doc_ids": ["html_b39f2948e2"],
        "expected_answer": "registrar@iiitn.ac.in",
        "is_answerable": True
    },
    {
        "query_id": "q4",
        "category": "OutOfScope",
        "query": "What is today's lunch menu in the IIIT Nagpur hostel mess?",
        "ground_truth_doc_ids": [],
        "expected_answer": "The provided IIIT Nagpur documents do not contain enough information to answer this question.",
        "is_answerable": False
    },
    {
        "query_id": "q5",
        "category": "OutOfScope",
        "query": "Who won the inter-hostel cricket match yesterday at IIIT Nagpur?",
        "ground_truth_doc_ids": [],
        "expected_answer": "The provided IIIT Nagpur documents do not contain enough information to answer this question.",
        "is_answerable": False
    }
]

# Ensure data directory exists and save eval dataset
os.makedirs(os.path.dirname(EVAL_DATASET_PATH), exist_ok=True)
with open(EVAL_DATASET_PATH, "w", encoding="utf-8") as f:
    json.dump(benchmark_data, f, indent=2)

print(f"Eval dataset saved successfully to: {EVAL_DATASET_PATH}")
print(f"Total Evaluation Benchmarks Loaded: {len(benchmark_data)}")

Eval dataset saved successfully to: E:\Rasengan\College-Knowledge-Search\data\eval_dataset.json
Total Evaluation Benchmarks Loaded: 5


In [10]:
available_doc_ids = {chunk["doc_id"] for chunk in chunks}

for item in benchmark_data:
    expected_ids = set(item["ground_truth_doc_ids"])
    missing_ids = expected_ids - available_doc_ids

    print(f"\nQuery ID: {item['query_id']}")
    print(f"Query: {item['query']}")
    print(f"Expected Document IDs: {expected_ids}")

    if missing_ids:
        print(f"ERROR: Missing document IDs: {missing_ids}")
    else:
        print("Ground Truth: Valid")

print("\n" + "=" * 60)
print("EVALUATION DATASET SUMMARY")
print("=" * 60)

print(f"Total Queries: {len(benchmark_data)}")
print(f"Answerable Queries: {sum(q['is_answerable'] for q in benchmark_data)}")
print(f"Unanswerable Queries: {sum(not q['is_answerable'] for q in benchmark_data)}")
print(f"Total Available Documents: {len(available_doc_ids)}")



Query ID: q1
Query: Who is the Director of IIIT Nagpur and what is his email?
Expected Document IDs: {'html_b39f2948e2'}
Ground Truth: Valid

Query ID: q2
Query: Who is the HOD of Computer Science and Engineering?
Expected Document IDs: {'pdf_fb94fbf05b774a7e80dfef4a125c4dcb'}
ERROR: Missing document IDs: {'pdf_fb94fbf05b774a7e80dfef4a125c4dcb'}

Query ID: q3
Query: What is the contact email for the Registrar of IIIT Nagpur?
Expected Document IDs: {'html_b39f2948e2'}
Ground Truth: Valid

Query ID: q4
Query: What is today's lunch menu in the IIIT Nagpur hostel mess?
Expected Document IDs: set()
Ground Truth: Valid

Query ID: q5
Query: Who won the inter-hostel cricket match yesterday at IIIT Nagpur?
Expected Document IDs: set()
Ground Truth: Valid

EVALUATION DATASET SUMMARY
Total Queries: 5
Answerable Queries: 3
Unanswerable Queries: 2
Total Available Documents: 314


In [11]:
import time
import pandas as pd
import numpy as np
from collections import defaultdict

TOP_K = 5

print("Evaluation environment initialized.")
print(f"Total benchmark queries: {len(benchmark_data)}")
print(f"Evaluation Top-K: {TOP_K}")


Evaluation environment initialized.
Total benchmark queries: 5
Evaluation Top-K: 5


In [12]:
def calculate_retrieval_metrics(retrieved_doc_ids, relevant_doc_ids, k=5):

    retrieved = retrieved_doc_ids[:k]
    relevant = set(relevant_doc_ids)

    if not relevant:
        return {
            f"Precision@{k}": np.nan,
            f"Recall@{k}": np.nan,
            "MRR": np.nan
        }

    relevant_retrieved = sum(
        1 for doc_id in retrieved if doc_id in relevant
    )

    precision = relevant_retrieved / k
    recall = relevant_retrieved / len(relevant)

    reciprocal_rank = 0.0

    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            reciprocal_rank = 1 / rank
            break

    return {
        f"Precision@{k}": precision,
        f"Recall@{k}": recall,
        "MRR": reciprocal_rank
    }


In [13]:
def get_unique_doc_ids(indices, top_k=TOP_K):
    doc_ids = []
    seen = set()

    for idx in indices:
        if idx < 0:
            continue

        doc_id = chunks[int(idx)]["doc_id"]

        if doc_id not in seen:
            seen.add(doc_id)
            doc_ids.append(doc_id)

        if len(doc_ids) == top_k:
            break

    return doc_ids


def retrieve_bm25_doc_ids(query, top_k=TOP_K):
    scores = bm25_index.get_scores(query.lower().split())
    ranked_indices = np.argsort(scores)[::-1]

    return get_unique_doc_ids(ranked_indices, top_k)


def retrieve_faiss_doc_ids(query, top_k=TOP_K):
    query_embedding = bi_encoder.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    _, indices = faiss_index.search(
        query_embedding,
        min(len(chunks), top_k * 10)
    )

    return get_unique_doc_ids(indices[0], top_k)


def retrieve_hybrid_doc_ids(query, top_k=TOP_K):
    results = hybrid_retrieve_and_rerank(
        query,
        candidate_k=20,
        final_top_k=20
    )

    doc_ids = []
    seen = set()

    for result in results:
        doc_id = result["doc_id"]

        if doc_id not in seen:
            seen.add(doc_id)
            doc_ids.append(doc_id)

        if len(doc_ids) == top_k:
            break

    return doc_ids


retrievers = {
    "BM25": retrieve_bm25_doc_ids,
    "FAISS": retrieve_faiss_doc_ids,
    "Hybrid": retrieve_hybrid_doc_ids
}

retrieval_results = []

for item in benchmark_data:
    if not item["is_answerable"]:
        continue

    for retriever_name, retrieve_fn in retrievers.items():
        retrieved_ids = retrieve_fn(item["query"])

        metrics = calculate_retrieval_metrics(
            retrieved_ids,
            item["ground_truth_doc_ids"],
            TOP_K
        )

        retrieval_results.append({
            "Query ID": item["query_id"],
            "Category": item["category"],
            "Retriever": retriever_name,
            **metrics
        })

retrieval_df = pd.DataFrame(retrieval_results)

display(retrieval_df)

,Query ID,Category,Retriever,Precision@5,Recall@5,MRR
0,q1,Institute,BM25,0.0,0.0,0.0
1,q1,Institute,FAISS,0.0,0.0,0.0
2,q1,Institute,Hybrid,0.0,0.0,0.0
3,q2,Academics,BM25,0.0,0.0,0.0
4,q2,Academics,FAISS,0.0,0.0,0.0
5,q2,Academics,Hybrid,0.0,0.0,0.0
6,q3,Admissions,BM25,0.0,0.0,0.0
7,q3,Admissions,FAISS,0.0,0.0,0.0
8,q3,Admissions,Hybrid,0.0,0.0,0.0


In [14]:
# Retrieval Evaluation Diagnosis

for item in benchmark_data:
    if not item["is_answerable"]:
        continue

    query = item["query"]
    expected_ids = item["ground_truth_doc_ids"]

    print(f"\n{'=' * 80}")
    print(f"Query: {query}")
    print(f"Expected document IDs: {expected_ids}")

    matching_chunks = [
        chunk for chunk in chunks
        if chunk["doc_id"] in expected_ids
    ]

    print(f"Expected IDs found in chunks: {len(matching_chunks)}")

    for retriever_name, retrieve_fn in retrievers.items():
        retrieved_ids = retrieve_fn(query)

        print(f"\n{retriever_name}:")
        for doc_id in retrieved_ids:
            matching = next(
                (c for c in chunks if c["doc_id"] == doc_id),
                None
            )

            title = matching["title"] if matching else "Unknown"
            print(f"  {doc_id} | {title}")

    print("-" * 80)


Query: Who is the Director of IIIT Nagpur and what is his email?
Expected document IDs: ['html_b39f2948e2']
Expected IDs found in chunks: 3

BM25:
  pdf_6f4e75b23b | Policy of Prevention of Sexual Harassment of Women at Workplace
  pdf_ed0072316b | Learn more
  pdf_0d8d169915 | Learn more
  pdf_2f287056d3 | 📄 Guide to RTI Act,2005(English)
  html_d9b7225407 | Indian Institute of Information Technology Nagpur

FAISS:
  pdf_8d3e6fb6bb | Read More →
  html_8b7f8b2192 | Indian Institute of Information Technology Nagpur
  html_6389c9ae2a | Indian Institute of Information Technology Nagpur
  pdf_4a7e1fedb9 | Read More →
  html_bd38537c1c | Indian Institute of Information Technology Nagpur

Hybrid:
  html_6389c9ae2a | Indian Institute of Information Technology Nagpur
  pdf_4a7e1fedb9 | Read More →
  pdf_8d3e6fb6bb | Read More →
  html_8b7f8b2192 | Indian Institute of Information Technology Nagpur
  html_d9b7225407 | Indian Institute of Information Technology Nagpur
--------------------------

In [15]:
# Inspect Ground-Truth Chunks

search_terms = [
    "director@iiitn.ac.in",
    "registrar@iiitn.ac.in",
    "Prem Lal Patel"
]

for term in search_terms:
    print(f"\n{'=' * 80}")
    print(f"Searching for: {term}")

    matches = [
        chunk for chunk in chunks
        if term.lower() in chunk["chunk_text"].lower()
    ]

    print(f"Matching chunks: {len(matches)}")

    for chunk in matches[:5]:
        print(f"\nDocument ID: {chunk['doc_id']}")
        print(f"Title: {chunk['title']}")
        print(f"Category: {chunk['category']}")
        print(f"Content:\n{chunk['chunk_text'][:1000]}")


Searching for: director@iiitn.ac.in
Matching chunks: 86

Document ID: pdf_d34d247be2
Title: Telephone Directory
Category: General
Content:
Document: Telephone Directory | Category: General | Source: https://iiitn.ac.in/media/fb94fbf05b774a7e80dfef4a125c4dcb.pdf

--- Page 1 --- sameer टेल(cid:547)फ़ोन डाइरे(cid:200)टर(cid:547) / Directory (cid:272)(cid:871)मक सं(cid:201)या नाम Name पद Designation मोबाइल नंबर इंटरकॉम नंबर ईमेल आईडी Email- ID Sr No. Mobile Number Intercom No. 1 (cid:292)ो (डॉ) (cid:292)ेमलाल पटेल / Prof (Dr) Prem Lal Patel (cid:467)नदेशक / Director 9904003857 309 director@iiitn.ac.in 2 डॉ मयुर पराते / Dr Mayur Parate (cid:292)भार(cid:547) कुलस(cid:876)चव / Incharge Registrar 7387439968 291 registrar@iiitn.ac.in tdiwan@iiitn.ac.in 3 डॉ तौसीफ (cid:465)दवान / Dr Tausif Diwan सहयोगी अ(cid:876)ध(cid:231)ठाता (cid:871)श(cid:162)ा(cid:874)व(cid:622) / Associate Dean Academic 9890280388 324 assodean_acd@iiitn.ac.in 4 डॉ आ(cid:467)तश दया(cid:91)पूरकर / Dr Aatish Daryapurkar सहयोगी